# Tutorial 2: Losses of IDER

This notebook shows how to use the losses in `script/losses.py` on a toy batch.

IDER trains the regressor with `loss = loss_DEG + PCCLoss_lambda * loss_PCC`.

* `PearsonCorrCoef_loss`: 1 - PCC between the predicted and the measured expression (per gene, the training code takes the mean)
* `Diff_Uscore` + `DEGrank_loss`: a differentiable Mann-Whitney U-score of the prediction and its PCC loss to the ground-truth U-score


In [1]:
import sys, warnings
warnings.filterwarnings("ignore", category=UserWarning)   # torchmetrics notice
sys.path.insert(0, "../script")

import numpy as np
import torch

from losses import PearsonCorrCoef_loss, Diff_Uscore, DEGrank_loss
from culc_DEG_metrics import DEG_ranking, DEG_metric_func

rng = np.random.default_rng(0)
_ = torch.manual_seed(0)

## 1. Toy batch

* `st`: measured expression of the batch `[B, G]`
* `cluster_ids`: cluster of every spot (k-means clusters in training)
* `y_pred`: the model output `[B, G]` (here a leaf tensor with a gradient, instead of a network)
* `Uscore_gt`: ground-truth U-scores `{cluster id: [G]}` of the whole slide. It is computed once before training with `DEG_ranking`.

Cluster `c` over-expresses the genes `5*c ... 5*c+4`.

In [2]:
B, G, n_clusters = 256, 30, 4
cluster_ids = torch.tensor(rng.integers(0, n_clusters, B))

st = torch.tensor(rng.normal(size=(B, G)), dtype=torch.float32)
for c in range(n_clusters):
    st[cluster_ids == c, 5 * c: 5 * c + 5] += 2.0

_, Uscore_gt = DEG_ranking(st, cluster_ids.numpy(), device="cpu")
y_pred = (st + 2.0 * torch.randn_like(st)).requires_grad_()   # a noisy prediction
slide_ids = np.array(["A"] * B)   # not used by the losses, kept for compatibility

## 2. Differentiable U-score: `Diff_Uscore`

`DEG_ranking` (the metric) uses ranks and cannot be differentiated. `Diff_Uscore` replaces the comparison `a > b` of the Mann-Whitney U by `sigmoid((a - b) / temperature)`, so the gradient flows to `y_pred`.
It returns `{cid: U-score [G]}` for the clusters in `sampling_cid`.

The smaller the temperature (`--temperature`, 0.01 by default), the closer it is to the exact U-score.

In [3]:
cid = 1
U_soft = Diff_Uscore(y_pred, cluster_ids, torch.tensor([cid]), temperature=0.01)
print("U-score of cluster", cid, ":", tuple(U_soft[cid].shape), "| requires grad:", U_soft[cid].requires_grad)

U-score of cluster 1 : (30,) | requires grad: True


## 3. DEG-rank loss: `DEGrank_loss`

`DEGrank_loss(Uscore_pred, Uscore_gt, cid)` is `1 - PCC` between the predicted and the ground-truth U-score of cluster `cid` (each is normalized by its sum).

In training (`IDER_script/train.py`), 2 clusters are sampled at random in every batch, and the loss is computed for **the last of the two sampled clusters**.
The default temperature is `0.01` (`--temperature`).

In [4]:
sampling_cid = torch.unique(cluster_ids)[torch.randperm(n_clusters)[:2]][-1:]   # same as in train.py
cid = sampling_cid.item()

Uscore_pred = Diff_Uscore(y_pred, cluster_ids, sampling_cid, temperature=0.01)
loss_DEG = DEGrank_loss(Uscore_pred, Uscore_gt, cid)
print("sampled cluster:", cid, "| loss_DEG:", loss_DEG.item())

sampled cluster: 1 | loss_DEG: 0.04583746510411135


## 4. IDER loss and gradient

`loss = loss_DEG + PCCLoss_lambda * loss_PCC` (`--PCCLoss_lambda`, 1.0 by default).

In [5]:
PCCLoss_lambda = 1.0
loss_PCC = PearsonCorrCoef_loss(y_pred, st, slide_ids).mean()
loss = loss_DEG + PCCLoss_lambda * loss_PCC
loss.backward()
print("loss_DEG %.4f + %.1f * loss_PCC %.4f = %.4f" % (loss_DEG.item(), PCCLoss_lambda, loss_PCC.item(), loss.item()))
print("gradient reaches y_pred:", tuple(y_pred.grad.shape), "| |grad| mean:", y_pred.grad.abs().mean().item())

loss_DEG 0.0458 + 1.0 * loss_PCC 0.4888 = 0.5346
gradient reaches y_pred: (256, 30) | |grad| mean: 8.814117609290406e-05


## 5. Does minimizing the DEG-rank loss improve the DEG metrics?

As a minimal check, we optimize the prediction `y` itself (instead of a network) with Adam using only `loss_DEG`, with a new random cluster every step,
and evaluate with `DEG_metric_func` before and after.

In [6]:
def deg_metrics(y):
    _, U_pred = DEG_ranking(y.detach(), cluster_ids.numpy(), device="cpu")
    res = DEG_metric_func(U_pred, Uscore_gt, topk_list=[5, 10])
    return {k: round(float(v), 3) for k, v in res.items()}

y = (st + 3.0 * torch.randn_like(st)).requires_grad_()
opt = torch.optim.Adam([y], lr=0.05)
print("before:", deg_metrics(y))

for step in range(300):
    sampling_cid = torch.unique(cluster_ids)[torch.randperm(n_clusters)[:2]][-1:]
    Uscore_pred = Diff_Uscore(y, cluster_ids, sampling_cid, temperature=0.01)
    loss_DEG = DEGrank_loss(Uscore_pred, Uscore_gt, sampling_cid.item())
    loss_DEG.backward()
    opt.step(); opt.zero_grad()
    if step % 100 == 0:
        print("step %3d | loss_DEG %.3f" % (step, loss_DEG.item()))

print("after :", deg_metrics(y))

before: {'spearman': 0.709, 'ndcgBinary@5': 1.0, 'ndcgBinary@10': 0.824}
step   0 | loss_DEG 0.091


step 100 | loss_DEG 0.000


step 200 | loss_DEG 0.000


after : {'spearman': 1.0, 'ndcgBinary@5': 1.0, 'ndcgBinary@10': 0.984}


In the real code, `y_pred` is the output of the ST-Net (`model.py`) whose input is the CONCH features of the patches, and the ground-truth U-scores `Uscore_train` are computed from the k-means clusters of each training slide (`dataloader.py`).